# Galerie de réentraînement — À TOI DE JOUER (régression santé)

> 🟠 **Notebook semi-guidé.** Contrairement aux notebooks 01 et 02 (entièrement
> résolus), ici **le code est à toi** : chaque cellule contient des `# TODO`.
> Tu **reconstruis** le pattern toi-même. C'est cette étape qui ancre vraiment C5.
>
> 📉 **Le guidage décroît** au fil du notebook : les premières étapes te tiennent
> un peu la main, les dernières (entraînement, évaluation) te laissent **juste
> l'objectif** — à toi de retrouver le comment.

**Ta référence** : le notebook `02_regression_immobilier.ipynb` (résolu) + la
fiche A4 `fiche_pattern_ML_supervise.md`. Transpose, **ne copie pas bêtement** —
le dataset est différent, à toi d'adapter.

> 💡 Si tu bloques plus de 10 min sur une étape, ouvre le 02 à l'étape
> correspondante. Le but n'est pas de souffrir, c'est de **refaire le geste**.

## 🎬 Situation

**Mardi 9h chez FastIA.** Karim te confie un dossier :

> **De :** Dr. Antoine Vasseur, recherche clinique — Institut Valmont (fictif)
> **Objet :** Prototype prédiction de progression du diabète
>
> Bonjour FastIA,
>
> Nous voulons estimer la **progression du diabète à un an** d'un patient à
> partir de mesures cliniques de base (âge, IMC, pression, bilans sanguins).
> Objectif : repérer tôt les patients à forte progression pour un suivi
> renforcé.
>
> Pouvez-vous nous entraîner un modèle de **régression**, comparer 2
> configurations — et surtout nous dire **si un modèle simple ne suffirait
> pas** ? On se méfie des usines à gaz.
>
> Cordialement, Dr. Vasseur.

**Dataset** : `sklearn.datasets.load_diabetes()` — embarqué (442 patients,
10 features cliniques **déjà normalisées**, cible = score de progression à 1 an).

## 🧭 Le pattern (le même qu'au 02)

`[1] Charger → [2] Explorer → [3] Découper → [4] Entraîner → [5] Évaluer → [6] Persister`

**Twist de ce cas** : à l'étape **[4 bis]**, on se donne **deux points de
comparaison** — un **plancher** (`DummyRegressor`, le « modèle bête ») et un
**modèle simple** (`Ridge`). C'est ce qui permet de répondre au Dr. Vasseur :
*est-ce qu'une usine à gaz est seulement justifiée ?*

## Setup

In [ ]:
# Imports fournis (ce n'est pas le point d'apprentissage ici)
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## [1] Charger

> 🎯 But : obtenir un `DataFrame` avec les features + une colonne `target`.
> Indice : `load_diabetes(as_frame=True)` expose `.frame` (features + target).

In [ ]:
# TODO 1 — charge le dataset et construis df (features + colonne 'target')
# data = load_diabetes(...)
# df = ...

# print(f"Lignes : {df.shape[0]} | Colonnes : {df.shape[1]}")
# df.head()

## [2] Explorer (mini, 10 min max)

> 🎯 But : 3 vérifs — manquants ? distribution de la cible ? une relation clé ?
> En régression on regarde la **cible continue** (pas l'équilibre de classes).

In [ ]:
# TODO 2 — vérifie les manquants + décris la cible
# print('Manquants :', ...)
# print(df['target'].describe())

# (optionnel) un graphe : distribution de la cible, ou cible vs 'bmi'

## [3] Découper train / test

> 🎯 But : split reproductible 80/20.
> ⚠️ **Question** : faut-il `stratify=y` ici ? (rappelle-toi le 02…)

In [ ]:
# TODO 3 — sépare X / y puis fais le train_test_split
# X = ...
# y = ...
# X_train, X_test, y_train, y_test = train_test_split(...)

# print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")

## [4] Entraîner — 2 configurations

> 🎯 But : 2 `RandomForestRegressor` aux réglages différents.
> ⚠️ **Cette fois, pas de scaler pour le RandomForest** : il est insensible à
> l'échelle (tu en mettras un pour le `Ridge` à l'étape [4 bis], lui en a
> besoin). Config A = défaut, Config B = un peu plus de capacité mais bridée
> (petit dataset, 442 lignes → attention à l'overfit).

In [ ]:
# TODO 4
# Entraîne 2 RandomForestRegressor aux réglages différents.
# Pas besoin de scaler pour un RandomForest (insensible à l'échelle).
# Config A = défaut. Config B = un peu plus de capacité, mais bridée
# (petit dataset, 442 lignes → attention à l'overfit).


## [4 bis] Les deux références : un plancher + un modèle simple

> 🎯 But : avant de juger tes RandomForest, donne-toi **deux points de
> comparaison**.
>
> - **Le plancher** — un `DummyRegressor(strategy="mean")` qui prédit toujours
>   la moyenne. Si ton RF ne le bat pas nettement, il n'apprend rien. C'est le
>   minimum vital.
> - **Le modèle simple** — un `Ridge` (régression linéaire régularisée). La
>   vraie question du Dr. Vasseur : *le RandomForest complexe fait-il mieux
>   qu'un Ridge simple ?*
>
> ⚠️ **Détail qui compte** : le **`Ridge` a besoin d'un `StandardScaler`** (un
> modèle linéaire est sensible à l'échelle) → mets-le dans un
> `Pipeline(StandardScaler → Ridge)`. Le `Dummy` et le `RandomForest`, **non**.

In [ ]:
# TODO 4 bis
# 1) Le plancher : un DummyRegressor(strategy="mean"), entraîné sur X_train/y_train.
# 2) Le modèle simple : un Ridge, mais DANS un Pipeline(StandardScaler -> Ridge)
#    — le linéaire a besoin du scaler ; le Dummy et le RandomForest, non.
# Tu compareras les 4 modèles (Dummy, Ridge, RF A, RF B) à l'étape [5].


## [5] Évaluer sur le test

> 🎯 But : évaluer les **4 modèles** (Dummy, Ridge, RF A, RF B) sur le jeu de
> test et les comparer dans un tableau qui permet de **choisir**. À toi de
> retrouver : *quelles métriques* pour une régression ? *comment* les stocker
> et les afficher côte à côte ? (Si tu sèches sur les métriques, regarde
> l'étape [5] du notebook 02.)

In [ ]:
# TODO 5
# Compare les 4 modèles (Dummy, Ridge, RF A, RF B) sur le jeu de test.
# Construis un tableau récapitulatif qui te permet de choisir le meilleur.
# (À toi de retrouver : quelles métriques de régression ? comment les
#  calculer, les stocker, les afficher côte à côte ?)


### ✅ Vérifie-toi (sans regarder de solution)

Sur ce dataset (split `random_state=42`), tu dois obtenir, **à peu près** :

| Modèle | MAE | RMSE | R² |
|---|---|---|---|
| **Plancher (Dummy)** | ~ 62 à 66 | ~ 72 à 74 | **≈ 0** (par construction) |
| **Ridge / RandomForest** | ~ 42 à 46 | ~ 53 à 55 | ~ **0,40 à 0,48** |

> ✅ Le **plancher** doit être **nettement battu** par tes vrais modèles (R² ≈ 0
> pour le Dummy → tes modèles ~0,45). S'ils ne le battent pas, cherche le bug.

> 😮 **R² « seulement » 0,45 ?!** Oui — et c'est **normal**. Diabetes est un
> dataset difficile : contrairement à l'immobilier californien (R² > 0,80),
> 10 mesures cliniques n'expliquent qu'une **partie** de la progression. Un R²
> bas n'est pas forcément un bug : parfois **le signal n'est pas dans les
> données**. C'est un constat de pro, pas un échec.

> 🔑 **Et le Ridge ?** Tu devrais voir que le **Ridge fait quasiment aussi
> bien** que le RandomForest (R² ~0,45 lui aussi). Leçon : sur un petit
> dataset au signal surtout linéaire, **le modèle complexe n'apporte rien**.
> C'est exactement ce qu'il faut répondre au Dr. Vasseur.

## [6] Persister le meilleur modèle

> 🎯 But : sauver le pipeline retenu (le plus simple **à performance égale** !).

In [ ]:
# TODO 6 — sauve le pipeline retenu avec joblib (compress=3) dans models/
# models_dir = Path('models'); models_dir.mkdir(exist_ok=True)
# joblib.dump(best_pipeline, models_dir / 'valmont_progression_v1.joblib', compress=3)

## 🔎 Ce que ce cas t'a fait travailler

- **Refaire le pattern de mémoire** (pas relire un notebook résolu).
- **Deux références, deux questions** : le **plancher** (Dummy) répond à « mon
  modèle apprend-il quelque chose ? » ; le **modèle simple** (Ridge) répond à
  « le modèle complexe est-il seulement justifié ? ». Ne les confonds pas.
- **Un R² bas peut être normal** — le diagnostic vient des données, pas
  forcément du modèle.
- **Scaler ou pas** : le `Ridge` (linéaire) en a besoin, le `RandomForest`
  (arbres) non. Savoir lequel met quoi est un réflexe de pro.

> ⭐ **Pour aller plus loin** : affiche les `feature_importances_` du
> RandomForest — quelles mesures cliniques pèsent le plus ? Et tente un
> `GradientBoostingRegressor` : gagne-t-il vraiment sur le Ridge ?

---

*Galerie de réentraînement — à toi de jouer (régression, Institut Valmont).*